# Hamiltonian Adaptive Ternary Tree

HATT is an algorithm for the construction of Ternary-Tree encodings which are optimised for the Hamiltonian of interest.

This notebook shows how to reproduce the results of: 
Y. Liu et al., "HATT: Hamiltonian Adaptive Ternary Tree for Optimizing Fermion-to-Qubit Mapping," 2025 IEEE International Symposium on High Performance Computer Architecture (HPCA), Las Vegas, NV, USA, 2025, pp. 143-157, doi: 10.1109/HPCA61900.2025.00022.

In [1]:
import pickle
from ferrmion.encode import TernaryTree
from pathlib import Path
from ferrmion.encode import TernaryTree
from ferrmion.encode.ternary_tree_node import TTNode
from pathlib import Path
import numpy as np

def water_integrals():
    folder = Path.cwd().joinpath(Path("../../../python/tests/"))
    with open(folder.joinpath("./data/water_1e.pkl"), 'rb') as file:
        ones = pickle.load(file)

    with open(folder.joinpath("./data/water_2e.pkl"), 'rb') as file:
        twos = pickle.load(file)
    return (ones, twos)

ones, twos = water_integrals()
# to avoid double-counting
twos = 0.5* twos

# Preprocess Hamiltonian

In [2]:
from itertools import product
import ferrmion as fr
def signature_char_to_ipowers(char:str):
    match char:
        case "+":
            return [1,-1j]
        case "-":
            return [1,1j]
        case "_":
            raise ValueError("Signature must contain only + and -")

def hamiltonian_term_to_majorana(majorana_ham, coeffs, signature):
    assert len(signature) == coeffs.ndim
    non_zero = np.where(coeffs != 0)
    non_zero_ones = [(*indices, coeffs[indices]) for indices in zip(*non_zero)]
    normalisation = 0.5**len(signature)

    
    ipowers = np.array([signature_char_to_ipowers(c) for c in signature])
    for *inds, coeff in non_zero_ones:
        # we need two majoranas for each fermionic operator
        left_right_indices = [[0,1]]*len(signature) 
        for left_right in product(*left_right_indices):
            majorana_ind = tuple([2*i+lr for i,lr in zip(inds, left_right)])
            term_ipowers = np.prod([ipow[lr] for ipow,lr in zip(ipowers, left_right)])
            majorana_ham[majorana_ind] = majorana_ham.get(majorana_ind, 0)
            majorana_ham[majorana_ind] += normalisation * coeff * term_ipowers

    return majorana_ham

def fermionic_to_majorana(hamiltonian_terms:list[tuple[np.ndarray, str]])-> dict[tuple[int],np.complex64]:
    total_ham = {}
    for coeffs, signature in hamiltonian_terms:
        total_ham.update(hamiltonian_term_to_majorana(total_ham, coeffs=coeffs, signature=signature))
    return total_ham
n_modes = ones.shape[0]
majorana_ham = fermionic_to_majorana([(ones, "+-"),(twos,"++--")])

# Algorithm 1

In [3]:
from itertools import combinations
majorana_ham = {(0,1):0.5j, (2,3):-0.5j, (4,5):-0.5j, (2,3,4,5):0.5}
n_modes = 3
# If any pauli term is found an even number f times, it returns I
# If we find all three pauli terms, return I (with an imaginary ccoefficient)
# If we find either one pauli or two then the weight is 1.

def term_weight(term, comb):
    # print(f"{term=}")
    # print(f"{comb=}")
    term_array = np.array([t for t in term])
    odd_parity_paulis = np.array([np.count_nonzero(np.array(term_array-index))%2 for index in comb])
    # print(f"{odd_parity_paulis=}")
    non_commuting = np.sum(odd_parity_paulis) % 3
    # print(f"{non_commuting=}")
    # print(term_array, comb, int(non_commuting!=0))
    return int(non_commuting!=0)

def reduce_hamiltonian(majorana_ham, parent_index, selection):
    new_ham = {}
    for term, coeff in majorana_ham.items():
        new_term = (i if i not in selection else parent_index for i in term)
        if len(set(new_term)) != 1:
            new_ham[new_term] = coeff
    return new_ham

nodes = {i:None for i in range(2*n_modes+1)}
unassigned = {*range(2*n_modes+1)}
for i in range(n_modes):
    parent_index = 2*n_modes+1+i
    print("Node index %s",parent_index)
    parent = TTNode(qubit_label=i)
    nodes[parent_index] = parent

    min = np.inf
    selection = None
    for comb in combinations(unassigned, 3):
        weight = np.sum([term_weight(term, comb) for term in majorana_ham.keys()])
        if weight < min:
            min = weight
            selection = comb

    print(selection, min)

    for (i,char) in zip(selection, ["x","y","z"]):
        unassigned.remove(i)
        if isinstance(nodes.get(i, None), TTNode):
            # print("Using TTNode as child %s", char)
            parent.add_child(which_child=char, child_node=nodes.get(i))
        else:
            parent.branch_majorana_indices[char] = i
    
    unassigned.add(parent_index)

    majorana_ham = reduce_hamiltonian(majorana_ham, parent_index, selection)

assert len(unassigned) == 1
print(unassigned)

root = nodes[unassigned.pop()]
root.branch_majorana_indices

Node index %s 7
(0, 1, 6) 1
Node index %s 8
(2, 3, 4) 0
Node index %s 9
(5, 7, 8) 0
{9}


{'x': 5, 'yx': 0, 'yz': 6, 'yy': 1, 'zx': 2, 'zz': 4, 'zy': 3}

# Algorithm 2

In [17]:
from itertools import combinations
majorana_ham = {(0,1):0.5j, (2,3):-0.5j, (4,5):-0.5j, (2,3,4,5):0.5}
n_modes = 3
# If any pauli term is found an even number f times, it returns I
# If we find all three pauli terms, return I (with an imaginary ccoefficient)
# If we find either one pauli or two then the weight is 1.

def term_weight(term, comb):
    # print(f"{term=}")
    # print(f"{comb=}")
    term_array = np.array([t for t in term])
    odd_parity_paulis = np.array([np.count_nonzero(np.array(term_array-index))%2 for index in comb])
    # print(f"{odd_parity_paulis=}")
    non_commuting = np.sum(odd_parity_paulis) % 3
    # print(f"{non_commuting=}")
    # print(term_array, comb, int(non_commuting!=0))
    return int(non_commuting!=0)

def reduce_hamiltonian(majorana_ham, parent_index, selection):
    new_ham = {}
    for term, coeff in majorana_ham.items():
        new_term = (i if i not in selection else parent_index for i in term)
        if len(set(new_term)) != 1:
            new_ham[new_term] = coeff
    return new_ham


nodes = {i:None for i in range(2*n_modes+1)}
for i in range(n_modes):
    nodes[2*n_modes+1+i] = TTNode(qubit_label=i)

unassigned = {*range(2*n_modes+1)}

for i in range(n_modes):
    parent_index = 2*n_modes+1+i
    parent = nodes[parent_index]

    min = np.inf
    selection = None
    for comb in combinations(unassigned, 2):
        small_y = None
        small_x = None
        x_index, z_index = comb

        x_node = nodes[x_index]
        if x_node is None:
            small_x = x_index
        elif isinstance(x_node, TTNode):
            small_x = x_node.z_descendant.branch_majorana_indices["z"]

        # discard this combination
        if small_x == 2*n_modes:
            continue

        if small_x % 2 == 0:
            small_y = small_x+1
        else:
            small_y = small_x-1
        # We can't use this index for y a 
        # it has been used in the combination already
        # so we'd be replacing our x or z!
        if small_y in comb:
            continue
        
        # If this leaf has not been used,
        # it can't have a Z-parent!
        if small_y in unassigned:
            y_index = small_y
        else:
            for node in nodes.values():
                if isinstance(node, TTNode) and node.branch_majorana_indices["z"]==small_y:
                    # Node indices are offset from qubit labels!
                    y_index = node.z_ancestor.qubit_label + 2*n_modes+1
                    break

        # print(f"{x_index=},{y_index=},{z_index=}")
        if small_x %2 ==0:
            comb = np.array([x_index, y_index, z_index], dtype=np.uint)
        else:
            comb = np.array([y_index, x_index, z_index], dtype=np.uint)
        comb = [int(i) for i in comb]
        # print(f"{comb=}")
        weight = np.sum([term_weight(term, comb) for term in majorana_ham.keys()])
        if weight < min:
            min = weight
            selection = comb


    # Now find the Y pair of the x-node


    for (i,char) in zip(selection, ["x","y","z"]):
        
        if i in unassigned:
            unassigned.remove(i)

        if isinstance(nodes.get(i, None), TTNode):
            parent.add_child(which_child=char, child_node=nodes.get(i))
        else:
            parent.branch_majorana_indices[char] = i
    
    unassigned.add(parent_index)
    majorana_ham = reduce_hamiltonian(majorana_ham, parent_index, selection)
assert len(unassigned) == 1

root = nodes[unassigned.pop()]
root.branch_majorana_indices

{'y': 5, 'xx': 2, 'xz': 4, 'xy': 3, 'zx': 0, 'zz': 6, 'zy': 1}